In [1]:
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

import timm
from sklearn.metrics import accuracy_score, f1_score

In [2]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

2.10.0+cu128
12.8
Tesla T4


In [18]:
CONFIG = {
    "img_size": 224,
    "batch_size": 32,
    "epochs": 30,
    "lr_head": 1e-3,
    "lr_full": 2e-5,
    "weight_decay": 0.05,
    "num_classes": 10,
    "num_workers": 4,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,
    "save_dir": "/kaggle/working/checkpoints",
    "dataset_path": "/kaggle/input/datasets/mahiruddin/skin-diseases-dataset-augmented/skin-diseases-dataset-augmented" 
}

In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

In [5]:
os.makedirs(CONFIG["save_dir"], exist_ok=True)

# Dataset Class

In [6]:
class SkinDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        
        for cls in self.classes:
            cls_path = os.path.join(root_dir, cls)
            for img_name in os.listdir(cls_path):
                self.samples.append((os.path.join(cls_path, img_name), self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Transforms

In [7]:
transform = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225))
])

# Train and Validation split

In [8]:
from sklearn.model_selection import train_test_split

dataset = SkinDataset(CONFIG["dataset_path"], transform=None)

indices = list(range(len(dataset)))
labels = [dataset.samples[i][1] for i in indices]

train_idx, temp_idx = train_test_split(
    indices, test_size=0.2, stratify=labels, random_state=CONFIG["seed"]
)

temp_labels = [labels[i] for i in temp_idx]

val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, stratify=temp_labels, random_state=CONFIG["seed"]
)

class SubsetDataset(Dataset):
    def __init__(self, dataset, indices, transform):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img_path, label = self.dataset.samples[self.indices[idx]]
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)
        return image, label

train_dataset = SubsetDataset(dataset, train_idx, transform)
val_dataset   = SubsetDataset(dataset, val_idx, transform)
test_dataset = SubsetDataset(dataset, test_idx, transform)

In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True
)

In [10]:
model = timm.create_model("deit3_base_patch16_224", pretrained=True)
model.head = nn.Linear(model.head.in_features, CONFIG["num_classes"])
model = model.to(CONFIG["device"])

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

In [11]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.cuda.amp.GradScaler()

/tmp/ipykernel_55/2280048282.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [12]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    
    for images, labels in tqdm(loader):
        images = images.to(CONFIG["device"])
        labels = labels.to(CONFIG["device"])
        
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

In [13]:
def validate(model, loader):
    model.eval()
    preds, targets = [], []
    total_loss = 0
    
    with torch.no_grad():
        for images, labels in tqdm(loader):
            images = images.to(CONFIG["device"])
            labels = labels.to(CONFIG["device"])
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            
            preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            targets.extend(labels.cpu().numpy())
    
    acc = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds, average='weighted')
    
    return total_loss / len(loader), acc, f1

In [14]:
for param in model.parameters():
    param.requires_grad = False

for param in model.head.parameters():
    param.requires_grad = True

optimizer = optim.AdamW(model.head.parameters(), lr=CONFIG["lr_head"])

In [15]:
print("Starting warm-up phase")

for epoch in range(5):
    train_loss = train_one_epoch(model, train_loader, optimizer)
    val_loss, val_acc, val_f1 = validate(model, val_loader)
    
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")

Starting warm-up phase


100%|██████████| 425/425 [02:39<00:00,  2.67it/s]


Epoch 1 | Train Loss: 1.1807 | Val Loss: 1.1132 | Acc: 0.7328 | F1: 0.7293


100%|██████████| 425/425 [02:39<00:00,  2.67it/s]


Epoch 2 | Train Loss: 1.1104 | Val Loss: 1.1022 | Acc: 0.7402 | F1: 0.7355


...

Epoch 3 | Train Loss: 1.0953 | Val Loss: 1.1041 | Acc: 0.7330 | F1: 0.7333


100%|██████████| 425/425 [02:39<00:00,  2.67it/s]


Epoch 4 | Train Loss: 1.0889 | Val Loss: 1.0870 | Acc: 0.7403 | F1: 0.7346


100%|██████████| 425/425 [02:39<00:00,  2.67it/s]

Epoch 5 | Train Loss: 1.0798 | Val Loss: 1.0698 | Acc: 0.7563 | F1: 0.7529


In [16]:
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.AdamW(model.parameters(), lr=CONFIG["lr_full"], weight_decay=CONFIG["weight_decay"])

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])

In [ ]:
best_acc = 0
train_losses = []
val_losses = []
val_accuracies = []

print("Starting full fine-tuning")

for epoch in range(CONFIG["epochs"]):
# for epoch in range(5):
    train_loss = train_one_epoch(model, train_loader, optimizer)
    val_loss, val_acc, val_f1 = validate(model, val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    
    scheduler.step()
    
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(CONFIG["save_dir"], "best_model.pth"))
    
    if (epoch + 1) % 5 == 0:
        torch.save(model.state_dict(), os.path.join(CONFIG["save_dir"], f"checkpoint_epoch_{epoch+1}.pth"))

Starting full fine-tuning


100%|██████████| 425/425 [02:38<00:00,  2.67it/s]


Epoch 1 | Train Loss: 1.2613 | Val Loss: 0.9625 | Acc: 0.7977 | F1: 0.7935


100%|██████████| 425/425 [02:39<00:00,  2.67it/s]


Epoch 2 | Train Loss: 0.8875 | Val Loss: 0.8765 | Acc: 0.8422 | F1: 0.8382


...

100%|██████████| 425/425 [02:38<00:00,  2.67it/s]


Epoch 23 | Train Loss: 0.5049 | Val Loss: 0.5830 | Acc: 0.9753 | F1: 0.9752


100%|██████████| 425/425 [02:39<00:00,  2.67it/s]


Epoch 24 | Train Loss: 0.5049 | Val Loss: 0.5827 | Acc: 0.9757 | F1: 0.9757


 14%|█▍        | 480/3395 [02:27<14:45,  3.29it/s]

In [ ]:
model.load_state_dict(torch.load(os.path.join(CONFIG["save_dir"], "best_model.pth")))
model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for images, labels in tqdm(test_loader):
        images = images.to(CONFIG["device"])
        labels = labels.to(CONFIG["device"])
        
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(labels.cpu().numpy())

from sklearn.metrics import accuracy_score, f1_score

test_acc = accuracy_score(all_targets, all_preds)
test_f1 = f1_score(all_targets, all_preds, average='weighted')

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_targets, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=dataset.classes,
            yticklabels=dataset.classes)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(all_targets, all_preds, target_names=dataset.classes))

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid()

plt.show()

In [ ]:
torch.save(model.state_dict(), os.path.join(CONFIG["save_dir"], "last_model.pth"))